# Week 39

In [ ]:
# !pip install -q --upgrade transformers datasets sacrebleu

In [ ]:
import pandas as pd
import re
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import nltk
from nltk.util import ngrams
import math
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk import FreqDist, ConditionalFreqDist
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch import nn
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader
from torch.optim import AdamW
from typing import List, Tuple
from transformers import MarianMTModel, MarianTokenizer, DistilBertTokenizerFast
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from tqdm import tqdm
import pandas as pd
import os
from transformers import DataCollatorForSeq2Seq
from datasets import Dataset
from transformers import Seq2SeqTrainer
import evaluate
from google.colab import drive
drive.mount('/content/drive')

## Load the dataset
splits = {'train': 'train.parquet', 'validation': 'validation.parquet'}
df_train = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["train"])
df_val = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["validation"])

# Only keep Arabic, Telugu and Korean examples
df_train = df_train[df_train['lang'].isin(['ar', 'te', 'ko'])]

df_train_te = df_train[df_train['lang'].isin(['te'])]

df_val_te = df_val[df_val['lang'].isin(['te'])]


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/andreasmelbye/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/andreasmelbye/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/Users/andreasmelbye/miniconda3/envs/nlp_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load the model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/Users/andreasmelbye/miniconda3/envs/nlp_env/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/Users/andreasmelbye/miniconda3/envs/nlp_env/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:5

In [ ]:
# --- Device ---
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Load multilingual mBART50 model ---
model_name = "facebook/mbart-large-50-many-to-many-mmt"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

In [ ]:
# --- Language mapping ---
LANG_MAP = {
    "te": "te_IN",   # Telugu
    "ar": "ar_AR",   # Arabic
    "ko": "ko_KR"    # Korean
}
TARGET_LANG = "te_IN"  # Telugu

# --- Translation function (English → Telugu) ---
def translate_to_telugu_mbart(text, source_lang="en_XX"):
    tokenizer.src_lang = source_lang  # English input
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)

    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id[TARGET_LANG],
        max_length=64,
        num_beams=5,
        early_stopping=True
    )

    translated_text = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
    return translated_text

# --- Example DataFrame ---
# df_train_te should have columns: ['question', 'context', 'answer']
# where 'answer' is in English

# Sti til hvor du vil gemme/indlæse
output_path = "/content/drive/MyDrive/train_with_telugu.csv"

if os.path.exists(output_path):
    print(f"Fandt eksisterende oversættelser i {output_path}, loader direkte...")
    df_train_te = pd.read_csv(output_path)
    print(df_train_te.head())
else:
    print("Ingen oversættelser fundet, kører translate_to_telugu_mbart ...")

    # Filtrer rows uden svar
    df_train_te = df_train_te[df_train_te['answer'].notnull()].reset_index(drop=True)

    # Oversæt svar til Telugu
    tqdm.pandas(desc="Translating English answers to Telugu")
    df_train_te['answer_inlang_trans'] = df_train_te['answer'].progress_apply(
        lambda x: translate_to_telugu_mbart(x)
    )
    df_train_te['answer_inlang'] = df_train_te['answer_inlang'].combine_first(df_train_te['answer_inlang_trans'])
    df_train_te = df_train_te.drop(columns=['answer_inlang_trans'])
    # Gem som CSV så vi ikke skal køre oversættelse igen
    df_train_te.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Oversættelser gemt til {output_path}")

# # Check some examples
# print(df_train_te[['question', 'answer', 'answer_inlang_trans','answerable']].head())

val_output_path = "/content/drive/MyDrive/val_with_telugu.csv"

if os.path.exists(val_output_path):
    print(f"Loading existing validation translations from {val_output_path}...")
    df_val_te = pd.read_csv(val_output_path)
else:
    print("Translating validation set...")
    tqdm.pandas(desc="Translating English answers to Telugu")
    df_val_te['answer_inlang_trans'] = df_val_te['answer'].progress_apply(
        lambda x: translate_to_telugu_mbart(x)
    )
    df_val_te['answer_inlang'] = df_val_te['answer_inlang'].combine_first(df_val_te['answer_inlang_trans'])
    df_val_te = df_val_te.drop(columns=['answer_inlang_trans'])
    df_val_te.to_csv(val_output_path, index=False, encoding="utf-8-sig")
    print(f"Saved translated validation set to {val_output_path}")

## Question + Context

In [ ]:
# # Get the question and context seperated by [SEP] token
# def get_question_context(row):
#     question = row['question']
#     context = row['context']
#     return question + " [SEP] " + context

# df_train_te['input_text'] = df_train_te.apply(get_question_context, axis=1)
# df_val_te['input_text'] = df_val_te.apply(get_question_context, axis=1)


# # df_train_te = df_train_te[['input_text', 'answer', 'answer_inlang','answerable']]
# df_val_te = df_val_te[['input_text', 'answer', 'answer_inlang','answerable']]

df_train_te.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Saved to {output_path}")

df_val_te.to_csv(val_output_path, index= False, encoding="utf-8-sig")

print(f"Saved validation to {val_output_path}")

/var/folders/mn/mlngcrqj5p90g299xqtw1qch0000gn/T/ipykernel_41322/3479198400.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train_te['input_text'] = df_train_te.apply(get_question_context, axis=1)
/var/folders/mn/mlngcrqj5p90g299xqtw1qch0000gn/T/ipykernel_41322/3479198400.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_val_te['input_text'] = df_val_te.apply(get_question_context, axis=1)


In [ ]:
# Make sure the column exists
df_train_te = df_train_te[['input_text', 'answer_inlang', 'answerable']] \
    .rename(columns={'answer_inlang': 'target_text'})

df_val_te = df_val_te[['input_text', 'answer_inlang', 'answerable']] \
    .rename(columns={'answer_inlang': 'target_text'})

# Convert to HF Dataset
train_dataset = Dataset.from_pandas(df_train_te, preserve_index=False)
val_dataset = Dataset.from_pandas(df_val_te, preserve_index=False)

max_input_length = 512   # depending on context length
max_target_length = 64   # answers are short

def tokenize_function(examples):
    # Tokenize inputs (question + context)
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=max_input_length,
        truncation=True
    )

    # Tokenize targets (Telugu answers)
    labels = tokenizer(
        examples["target_text"],
        max_length=max_target_length,
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

train_dataset = Dataset.from_pandas(df_train_te)
val_dataset = Dataset.from_pandas(df_val_te)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)


Map: 100%|██████████| 100/100 [00:00<00:00, 611.61 examples/s]


In [ ]:
# !pip install evaluate sacrebleu

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Load metrics
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
chrf_metric = evaluate.load("chrf")

def compute_metrics(eval_preds):
    """
    eval_preds: tuple (predictions, labels)
    - predictions: token ids
    - labels: token ids (with -100 for padding)
    """
    predictions, labels = eval_preds

    # Decode predictions and labels
    preds_text = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels_text = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Exact match
    em_scores = [int(pred.strip() == ref.strip()) for pred, ref in zip(preds_text, labels_text)]
    exact_match = np.mean(em_scores) * 100

    # BLEU
    bleu_result = bleu_metric.compute(predictions=preds_text, references=[[l] for l in labels_text])
    bleu_score = bleu_result["bleu"] * 100

    # ROUGE-L
    rouge_result = rouge_metric.compute(predictions=preds_text, references=labels_text)
    rouge_l = rouge_result["rougeL"] * 100

    # chrF
    chrf_result = chrf_metric.compute(predictions=preds_text, references=labels_text)
    chrf_score = chrf_result["score"] * 100

    return {
        "exact_match": exact_match,
        "bleu": bleu_score,
        "rougeL": rouge_l,
        "chrF": chrf_score
    }

In [ ]:
import transformers
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5-te-qa",
    # evaluation_strategy="epoch",
    # evaluate_during_training=True,
    label_smoothing_factor=0.1,
    learning_rate=1e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=5,
    predict_with_generate=True,
    logging_dir="./logs",
    logging_steps=50,
    save_safetensors=False
)

/Users/andreasmelbye/miniconda3/envs/nlp_env/lib/python3.12/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# n = 100
# shuffled = tokenized_train.shuffle(seed=42)

# tokenized_train = shuffled.select(range(n))

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

/Users/andreasmelbye/miniconda3/envs/nlp_env/lib/python3.12/site-packages/transformers/generation/utils.py:1258: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  elif isinstance(generation_config._eos_token_tensor, torch.Tensor):













                                             

                                         
100%|██████████| 7/7 [15:44<00:00, 43.48s/it]

                                             
100%|██████████| 7/7 [07:27<00:00, 63.99s/it]

{'eval_loss': 15.63286304473877, 'eval_bleu': 0.07847072435032314, 'eval_runtime': 49.94, 'eval_samples_per_second': 2.002, 'eval_steps_per_second': 0.26, 'epoch': 1.0}
{'train_runtime': 447.9302, 'train_samples_per_second': 0.112, 'train_steps_per_second': 0.016, 'train_loss': 20.599857875279017, 'epoch': 1.0}


TrainOutput(global_step=7, training_loss=20.599857875279017, metrics={'train_runtime': 447.9302, 'train_samples_per_second': 0.112, 'train_steps_per_second': 0.016, 'total_flos': 13373649408000.0, 'train_loss': 20.599857875279017, 'epoch': 1.0})

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

for example in tokenized_val.select(range(20)):
    input_text = example['input_text']
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512
    )

    # Move input tensors to same device as model
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Generate
    outputs = model.generate(**inputs, max_length=64, num_beams=5, early_stopping=True)

    print("Question + Context:", input_text)
    print("Generated Answer:", tokenizer.decode(outputs[0], skip_special_tokens=True))
    print("Reference:", example['target_text'])
    print("-----")

In [ ]:
# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

test_q = "బంగ్లాదేశ్ దేశ జాతీయ భాష ఏది"  # Example Telugu question
test_ctx = "English is a co-official language of Bangladesh and is widely used in the executive, legislative and judicial branches. Bangladesh's Constitution and laws were written in English and are now being re-written in the Bengali. It is also widely used in schools, colleges and universities as a medium of instruction"
input_text = test_q + " [SEP] " + test_ctx

inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}  # move tensors to same device
outputs = model.generate(**inputs, max_length=64)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
results = trainer.evaluate()

print(results)

df_val_te

In [ ]:
# --- General dataset stats ---
n_train = len(tokenized_train)
n_val = len(tokenized_val)

print(f"Training set size: {n_train}")
print(f"Validation set size: {n_val}")

# --- Count answerable vs unanswerable in val ---
n_val_answerable = sum(tokenized_val['answerable'])
n_val_unanswerable = n_val - n_val_answerable

print(f"Validation Answerable:   {n_val_answerable}")
print(f"Validation Unanswerable: {n_val_unanswerable}")

# --- Same for train ---
n_train_answerable = sum(tokenized_train['answerable'])
n_train_unanswerable = n_train - n_train_answerable

print(f"Training Answerable:   {n_train_answerable}")
print(f"Training Unanswerable: {n_train_unanswerable}")


val_answerable = tokenized_val.filter(lambda x: x["answerable"] == True)
val_unanswerable = tokenized_val.filter(lambda x: x["answerable"] == False)

results_answerable = trainer.evaluate(eval_dataset=val_answerable)
results_unanswerable = trainer.evaluate(eval_dataset=val_unanswerable)

print("Answerable:", results_answerable)
print("Unanswerable:", results_unanswerable)